In [9]:
import os
import json
import urllib.request
import urllib.parse
import streamlit as st
from dotenv import load_dotenv

from langchain_core.tools import tool
from langchain_core.messages import ToolMessage, HumanMessage, AIMessage
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper, ArxivAPIWrapper
from langchain_groq import ChatGroq

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import MessagesState
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt, Command
import wikipedia

In [10]:
load_dotenv()

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGSMITH_API_KEY")
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGSMITH_PROJECT")

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

llm = ChatGroq(model="openai/gpt-oss-120b")

In [11]:

load_dotenv()

@tool
def get_live_weather(city: str) -> str:
    """Fetch real-time weather data for a specific city name."""
    encoded_city = urllib.parse.quote(city)
    geo_url = f"https://geocoding-api.open-meteo.com/v1/search?name={encoded_city}&count=1&language=en&format=json"
    try:
        with urllib.request.urlopen(geo_url) as geo_response:
            geo_data = json.loads(geo_response.read().decode())
            if not geo_data.get("results"):
                return f"Could not find coordinates for city: {city}"
            loc = geo_data["results"][0]
            lat, lon, resolved = loc["latitude"], loc["longitude"], loc["name"]

        weather_url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current_weather=true"
        with urllib.request.urlopen(weather_url) as weather_response:
            weather_data = json.loads(weather_response.read().decode())
            current = weather_data["current_weather"]
            return f"Temperature in {resolved} is {current['temperature']}°C."
    except Exception as e:
        return f"Weather API Error: {str(e)}"

@tool
def search_arxiv(query: str) -> str:
    """Search the ArXiv database. DO NOT use conversational language. Use keywords or prefixes (e.g., 'all:quantum')."""
    try:
        arxiv = ArxivAPIWrapper(top_k_results=3, doc_content_chars_max=1500)
        result = arxiv.run(query)
        if not result or "No good Arxiv Result" in result:
            return f"Search failed. The query '{query}' returned no results. Try broader keywords."
        return result
    except Exception as e:
        return f"ArXiv API Error: {str(e)}"

tools = [search_arxiv, get_live_weather, WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())]

In [16]:
llm = ChatGroq(model="qwen/qwen3.6-27b") 
t_llm = llm.bind_tools(tools)
tool_node = ToolNode(tools)

def botnode(state: MessagesState):
    response = t_llm.invoke(state["messages"])
    return {"messages": [response]}

def approve_execution(state: MessagesState):
    last_message = state["messages"][-1]
    decision = interrupt({
        "step": "pre_execution",
        "action": "Approve Tool Run?",
        "tool_calls": last_message.tool_calls
    })

    if not decision.get("approved", False):
        rejections = [
            ToolMessage(
                tool_call_id=call["id"],
                content=f"Human rejected action: {decision.get('reason', 'Denied')}"
            ) for call in last_message.tool_calls
        ]
        return {"messages": rejections}
    return {}

def approve_result(state: MessagesState):
    last_message = state["messages"][-1]
    decision = interrupt({
        "step": "post_execution",
        "action": "Review Tool Result",
        "raw_result": last_message.content
    })

    if decision.get("edited_result"):
        edited_msg = ToolMessage(
            content=decision["edited_result"],
            tool_call_id=last_message.tool_call_id,
            id=last_message.id 
        )
        return {"messages": [edited_msg]}
    return {}


In [17]:

def route_after_approval(state: MessagesState):
    last_message = state["messages"][-1]
    return "botnode" if isinstance(last_message, ToolMessage) else "tools"

workflow = StateGraph(MessagesState)
workflow.add_node("botnode", botnode)
workflow.add_node("approve_execution", approve_execution)
workflow.add_node("tools", tool_node)
workflow.add_node("approve_result", approve_result)

workflow.add_edge(START, "botnode")
workflow.add_conditional_edges("botnode", tools_condition, {"tools": "approve_execution", "__end__": END})
workflow.add_conditional_edges("approve_execution", route_after_approval, {"botnode": "botnode", "tools": "tools"})
workflow.add_edge("tools", "approve_result")
workflow.add_edge("approve_result", "botnode")

memory = MemorySaver()
app = workflow.compile(checkpointer=memory)

In [ ]:
config = {"configurable": {"thread_id": "research-session-1"}}

# This will run until the LLM decides to call a tool, then pause.
initial_run = app.invoke({"messages": [("user", "Find a recent paper on deep learning architectures.")]}, config=config)

current_state = app.get_state(config)
print("Next step:", current_state.next)
print("Interrupt Payload:", current_state.tasks[0].interrupts[0].value)

Next step: ('approve_execution',)
Interrupt Payload: {'step': 'pre_execution', 'action': 'Approve Tool Run?', 'tool_calls': [{'name': 'search_arxiv', 'args': {'query': 'all:deep learning architectures'}, 'id': 'c5bs4f2zf', 'type': 'tool_call'}]}


In [19]:
# Pass True to authorize the network request
execution_approved = app.invoke(Command(resume={"approved": True}), config=config)

current_state = app.get_state(config)
print("Next step:", current_state.next)
print("Interrupt Payload:", current_state.tasks[0].interrupts[0].value)

Next step: ('approve_result',)
Interrupt Payload: {'step': 'post_execution', 'action': 'Review Tool Result', 'raw_result': "ArXiv API Error: 'Search' object has no attribute 'results'"}


In [20]:
# Accept the raw data as-is, or pass "edited_result": "Cleaned up string..."
final_run = app.invoke(Command(resume={"edited_result": None}), config=config)

print("\n--- Final Agent Response ---")
print(final_run["messages"][-1].content)


--- Final Agent Response ---



In [ ]:
rsp.pretty_print()

================================== Ai Message ==================================
Tool Calls:
  search_arxiv (fc_e0adf9b9-12b7-4cca-a9a9-7bbde091f2d1)
 Call ID: fc_e0adf9b9-12b7-4cca-a9a9-7bbde091f2d1
  Args:
    query: Attention Is All You Need


In [4]:
import os 
from dotenv import load_dotenv

import json 
import urllib.request
import urllib.parse

from langchain_core.tools import tool
from langchain_groq import ChatGroq
from langchain_core.messages import AIMessage,HumanMessage,ToolMessage
from langchain_community.utilities import WikipediaAPIWrapper,arxiv,ArxivAPIWrapper

from langgraph.graph.message import MessagesState
from langgraph.graph import StateGraph,END,START
from langgraph.prebuilt import tool_node,tools_condition
from langgraph.types import Command,interrupt
from langgraph.checkpoint.memory import MemorySaver

import wikipedia




In [5]:
load_dotenv()

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGSMITH_API_KEY")
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGSMITH_PROJECT")

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

llm = ChatGroq(model="openai/gpt-oss-120b")

In [ ]:

wiki_call=wikipedia(WikipediaAPIWrapper())
@tool 
def getweather_api():
    return 
@tool 
def arxiv():
    return 
@tool
def human_assit ():
    return 


tool=[wiki_call,getweather_api,arxiv,human_assit]





